In [1]:
import sys
from pathlib import Path

# 添加项目根目录到 Python 路径
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import os
import json
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Dict
from src import DsExtractor, QwenEvaluator, parse_response, DataPreprocessor


class AgentState(TypedDict):
    # 原始文本内容
    file_content: str
    # 来源标记：建议使用“文件路径 + 原文摘要”而非仅文件名
    source_reference: str
    # 对应你 json 中的 triplets 列表
    current_triplets: List[Dict]
    # Agent B 的逻辑审计反馈
    evaluation_feedback: List[str]
    # 质量评分，用于决定是否结束循环
    evaluation_score: float
    # 当前迭代次数
    iteration_count: int


改造大模型A-Extractor为图节点

In [ ]:
def extractor_node(state: AgentState):
    # 实例化你现有的 DsExtractor
    extractor = DsExtractor()

    # 构造动态 Prompt
    feedback_text = ""
    if state["iteration_count"] > 0:
        # 将评估反馈统一转换为文本，兼容字符串列表和字典列表
        feedback_items = []
        for item in state.get("evaluation_feedback", []):
            if isinstance(item, dict):
                item_type = item.get("type", "未分类问题")
                description = item.get("description", "")
                suggestion = item.get("suggestion", "")
                parts = [f"- {item_type}: {description}"]
                if suggestion:
                    parts.append(f"  建议: {suggestion}")
                feedback_items.append("\n".join(parts))
            else:
                feedback_items.append(str(item))

        # 核心：将 B 的逻辑审计意见强制加入下一轮的输入
        feedback_text = f"\n\n### 修正要求：\n" + "\n".join(feedback_items)

    # 来源标记优先使用状态中的动态来源（文件路径+原文证据），再兜底
    source_reference = state.get("source_reference", "unknown_source")

    # 调用你原来的抽取逻辑
    # 注意：这里的 text 融合了反馈，强迫模型针对性地补充中间节点或修正与门逻辑
    json_result = extractor.extract(
        text=state["file_content"] + feedback_text,
        source_reference=source_reference
    )

    # 解析 JSON 并更新状态
    triplets = parse_response(json_result) # 这里的解析逻辑保持你原有的

    return {
        "current_triplets": triplets,
        "iteration_count": state["iteration_count"] + 1
    }


编写大模型B-Evaluator为图节点

In [ ]:
def evaluator_node(state: AgentState):
    """将 QwenEvaluator 封装为 LangGraph 节点"""

    # 1. 实例化评估器
    evaluator = QwenEvaluator()

    # 2. 如果当前没有提取到三元组，直接判 0 分
    if not state.get("current_triplets"):
        return {
            "evaluation_score": 0.0,
            "evaluation_feedback": ["未提取到任何三元组，请重新检查文本。"]
        }

    try:
        # 3. 调用 AI 进行逻辑审计
        # 传入原始文本和当前提取出的三元组列表
        print(f"[Evaluator] 开始评估 {len(state['current_triplets'])} 个三元组...")
        json_result = evaluator.evaluate(
            text=state["file_content"],
            triplets=state["current_triplets"]
        )
        print(f"[Evaluator] 原始响应: {json_result[:200]}...")  # 只显示前200字符

        # 4. 解析审计报告
        report = json.loads(json_result)
        score = report.get("score", 0.0)
        feedback = report.get("missing_logic_details", [])
        if not isinstance(feedback, list):
            feedback = [str(feedback)]
        print(f"[Evaluator] 得分: {score}, 报告键: {report.keys()}")

        # 5. 返回更新后的状态
        return {
            "evaluation_score": score,
            "evaluation_feedback": feedback,
            "feedback_to_extractor": report.get("advice", "")
        }

    except Exception as e:
        # 容错处理：详细打印异常信息
        print(f"[Evaluator] 异常: {type(e).__name__}: {str(e)}")
        return {
            "evaluation_score": 0.0,
            "evaluation_feedback": [f"审计节点运行异常: {str(e)}"]
        }

In [ ]:
def decide_to_end(state: AgentState):
    # 打印当前进度
    print(f"[迭代{state.get('iteration_count', 0)}] 得分: {state['evaluation_score']:.2f}")
    
    # 如果模型报错导致没有 triplets，retry（最多2次）
    if not state.get("current_triplets") and state.get("iteration_count", 0) < 2:
        return "continue"

    # 降低迭代上限至 2（原来是 3）+ 提高评分阈值至 0.75（原来是 0.8）
    if state["evaluation_score"] >= 0.75 or state.get("iteration_count", 0) >= 2:
        return "end"

    return "continue"

In [ ]:
def create_workflow():
    """创建 LangGraph 工作流"""
    # 实例化图
    workflow = StateGraph(AgentState)

    # 注册节点
    workflow.add_node("extractor", extractor_node)
    workflow.add_node("evaluator", evaluator_node)

    # 设置起点
    workflow.set_entry_point("extractor")

    # 普通边：抽取器完成后总是去评估器
    workflow.add_edge("extractor", "evaluator")

    # 条件边：根据评估结果决定是否继续
    workflow.add_conditional_edges(
        "evaluator",
        decide_to_end,
        {
            "continue": "extractor",  # 继续抽取
            "end": END                # 结束流程
        }
    )

    # 编译工作流
    return workflow.compile()

In [3]:
def read_text_file(file_path: str) -> str:
    """统一读取 txt/pdf/图片，返回文本内容"""
    # 清洗输入路径：去掉首尾空白、引号，展开 ~，转换为绝对路径
    cleaned = file_path.strip().strip('"').strip("'")
    abs_path = os.path.abspath(os.path.expanduser(cleaned))

    print(f"[read_text_file] 输入路径: {repr(file_path)}")
    print(f"[read_text_file] 清洗后路径: {abs_path}")

    if not os.path.exists(abs_path):
        print(f"文件不存在: {abs_path}")
        return ""

    if os.path.isdir(abs_path):
        print(f"路径是目录而不是文件: {abs_path}")
        return ""

    ext = os.path.splitext(abs_path)[1].lower()

    # txt 文件保留编码兜底
    if ext == ".txt":
        for encoding in ("utf-8", "gb18030"):
            try:
                with open(abs_path, 'r', encoding=encoding) as f:
                    content = f.read()
                print(f"[read_text_file] 读取成功，类型: txt, 编码: {encoding}, 长度: {len(content)}")
                return content
            except UnicodeDecodeError:
                continue
            except Exception as e:
                print(f"读取 txt 失败: {e}")
                return ""

        print("读取失败: txt 编码既不是 utf-8 也不是 gb18030")
        return ""

    # pdf/图片等统一走预处理器
    try:
        content = DataPreprocessor.process_file(abs_path)
        if content is None:
            print(f"[read_text_file] 读取结果为空: {abs_path}")
            return ""

        content = str(content)
        print(f"[read_text_file] 读取成功，类型: {ext or 'unknown'}, 长度: {len(content)}")
        return content
    except Exception as e:
        print(f"读取文件失败: {e}")
        return ""

In [6]:
def save_results(triplets: list, output_file: str):
    """保存三元组结果到文件"""
    import json
    try:
        with open(output_file, 'w', encoding='utf-8') as f:
            json.dump({"triplets": triplets}, f, ensure_ascii=False, indent=2)
        print(f"结果已保存到: {output_file}")
    except Exception as e:
        print(f"保存结果失败: {e}")

In [4]:
        # 尝试从环境文件读取
        env_file = ".env"
        if os.path.exists(env_file):
            try:
                with open(env_file, 'r') as f:
                    for line in f:
                        if line.startswith("DASHSCOPE_API_KEY"):
                            key = line.split('=', 1)[1].strip()
                            os.environ["DASHSCOPE_API_KEY"] = key
                            print("已从 .env 文件读取 API 密钥")
                            break
            except Exception:
                pass
        if not os.getenv("DASHSCOPE_API_KEY"):
            print("警告: 未设置 DASHSCOPE_API_KEY 环境变量")
            print("请设置: export DASHSCOPE_API_KEY=your_api_key")

In [ ]:
# 输入文件路径
input_file = input("请输入文本文件路径 (默认: test.txt): ").strip()
if not input_file:
    input_file = "test.txt"

print(f"原始输入路径: {repr(input_file)}")

# 读取文件内容
file_content = read_text_file(input_file)
if not file_content:
    print("文件内容为空，程序退出")
    raise ValueError("输入文件为空或无法读取，请检查路径和编码")

print(f"文件内容长度: {len(file_content)} 字符\n")

# 构造更可信的来源标记：文件路径 + 原文首句摘要（避免只依赖文件名）
non_empty_lines = [line.strip() for line in file_content.splitlines() if line.strip()]
content_hint = non_empty_lines[0][:40] if non_empty_lines else file_content[:40]
source_reference = f"{input_file} | 摘要:{content_hint}"

# 创建初始状态（初始化所有必填字段）
initial_state = {
    "file_content": file_content,
    "source_reference": source_reference,
    "current_triplets": [],
    "evaluation_feedback": [],
    "evaluation_score": 0.0,
    "iteration_count": 0
}

# 创建并运行工作流
app = create_workflow()

print("开始运行 LangGraph 工作流...\n")

try:
    # 运行工作流
    final_state = app.invoke(initial_state)

    print("\n=== 抽取完成 ===")
    print(f"来源标记: {source_reference}")
    print(f"总迭代次数: {final_state['iteration_count']}")
    print(f"最终得分: {final_state['evaluation_score']}")
    print(f"抽取到 {len(final_state['current_triplets'])} 个三元组")

    # 显示三元组
    if final_state['current_triplets']:
        print("\n=== 抽取的三元组 ===")
        for i, triplet in enumerate(final_state['current_triplets'], 1):
            print(f"{i}. {triplet.get('subject_name', 'N/A')} -> {triplet.get('relation', 'N/A')} -> {triplet.get('object_name', 'N/A')}")
            print(f"   类型: {triplet.get('subject_type', 'N/A')} -> {triplet.get('object_type', 'N/A')}")
            print(f"   置信度: {triplet.get('confidence', 'N/A')}, 来源: {triplet.get('source', 'N/A')}\n")

    # 保存结果
    output_file = input("请输入结果保存路径 (默认: results.json): ").strip()
    if not output_file:
        output_file = "results.json"

    save_results(final_state['current_triplets'], output_file)

except Exception as e:
    print(f"工作流运行失败: {e}")


定义大模型A-Extractor和B-Evaluator的图节点关系

In [ ]:
# 1. 实例化图（确保 AgentState 已经定义，包含 triplets, score, feedback 等）
workflow = StateGraph(AgentState)

# 2. 注册节点
# 提示：确保你的 extractor_node 和 evaluator_node 接收 state 并返回字典
workflow.add_node("extractor", extractor_node)
workflow.add_node("evaluator", evaluator_node)

# 3. 设置起点
workflow.set_entry_point("extractor")

# 4. 普通边：A 完了总是要去 B
workflow.add_edge("extractor", "evaluator")

# 5. 条件边逻辑
def decide_to_end(state: AgentState):
    # 增加一个小逻辑：如果模型报错导致没有 triplets，也应该 retry 而不是直接报错
    if not state.get("current_triplets") and state.get("iteration_count", 0) < 3:
        return "continue"

    # 你的核心判断逻辑
    if state["evaluation_score"] >= 0.8 or state.get("iteration_count", 0) >= 3:
        return "end"

    return "continue"

# 6. 绑定条件边
workflow.add_conditional_edges(
    "evaluator",            # 哪一个节点发出的条件判断
    decide_to_end,          # 判断函数
    {
        "continue": "extractor", # 对应返回 "continue" 时跳回 extractor
        "end": END               # 对应返回 "end" 时结束
    }
)

# 7. 编译
app = workflow.compile()

仅使用 Extractor 抽取（不经过 Evaluator）并保存结果到固定目录。

In [ ]:
# 仅用 Extractor 抽取（不检验）

# 1) 输入文件路径
input_file = input("请输入文本文件路径 (默认: test.txt): ").strip()
if not input_file:
    input_file = "test.txt"

print(f"原始输入路径: {repr(input_file)}")

# 2) 读取文件内容
file_content = read_text_file(input_file)
if not file_content:
    raise ValueError("输入文件为空或无法读取，请检查路径和编码")

print(f"文件内容长度: {len(file_content)} 字符")

# 3) 构造来源标记（文件绝对路径 + 原文摘要）
cleaned_input = input_file.strip().strip('"').strip("'")
abs_input_path = os.path.abspath(os.path.expanduser(cleaned_input))
non_empty_lines = [line.strip() for line in file_content.splitlines() if line.strip()]
content_hint = non_empty_lines[0][:40] if non_empty_lines else file_content[:40]
source_reference = f"{abs_input_path} | 摘要:{content_hint}"

# 4) 调用 Extractor（不经过 Evaluator）
extractor = DsExtractor()
json_result = extractor.extract(
    text=file_content,
    source_reference=source_reference
)

# 5) 解析结果
triplets = parse_response(json_result)
print(f"Extractor 抽取完成，得到 {len(triplets)} 个三元组")

# 6) 保存到固定目录，并为同一输入文件自动递增版本号
output_dir = "/Users/oyster/PycharmProjects/FTP/data/output"
os.makedirs(output_dir, exist_ok=True)

base_name = os.path.splitext(os.path.basename(abs_input_path))[0]
prefix = f"{base_name}_triplets_v"
suffix = ".json"
latest_version = 0

for existing_name in os.listdir(output_dir):
    if not existing_name.startswith(prefix) or not existing_name.endswith(suffix):
        continue

    version_text = existing_name[len(prefix):-len(suffix)]
    if version_text.isdigit():
        latest_version = max(latest_version, int(version_text))

output_file = os.path.join(output_dir, f"{base_name}_triplets_v{latest_version + 1}.json")

with open(output_file, "w", encoding="utf-8") as f:
    json.dump({"triplets": triplets}, f, ensure_ascii=False, indent=2)

print(f"结果已保存到: {output_file}")

# 7) 简要预览
for i, t in enumerate(triplets[:5], 1):
    print(f"{i}. {t.get('subject_name', 'N/A')} -> {t.get('relation', 'N/A')} -> {t.get('object_name', 'N/A')}")
if len(triplets) > 5:
    print(f"... 其余 {len(triplets) - 5} 条已写入文件")

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

原始输入路径: '/Users/oyster/PycharmProjects/FTP/data/input/test2.pdf'
[read_text_file] 输入路径: '/Users/oyster/PycharmProjects/FTP/data/input/test2.pdf'
[read_text_file] 清洗后路径: /Users/oyster/PycharmProjects/FTP/data/input/test2.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could not get FontBBox from font descriptor because No

[read_text_file] 读取成功，类型: .pdf, 长度: 15182
文件内容长度: 15182 字符


KeyboardInterrupt: 